In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configuración
import warnings
warnings.filterwarnings('ignore')
plt.style.use('default')
np.random.seed(42)
#Modelo 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [17]:
#Importación de datos
df_stroke = pd.read_csv("healthcare-dataset-stroke-data.csv")

In [18]:
df_stroke = df_stroke.drop(columns=['id'])

In [19]:
X = df_stroke.drop('stroke', axis=1)
y = df_stroke['stroke']

In [20]:
#Train/Test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [21]:
#Separación de las variables
num_cols = ['age', 'avg_glucose_level', 'bmi']
cat_cols = [
    'gender', 'ever_married', 'work_type',
    'Residence_type', 'smoking_status'
]
bin_cols = ['hypertension', 'heart_disease']

In [22]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
bin_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

In [23]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols),
    ('bin', bin_pipeline, bin_cols)
])

In [24]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5001")
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://127.0.0.1:5001'


In [25]:
mlflow.search_experiments()

[<Experiment: artifact_location='file:///C:/Users/Lorena/Desktop/ULore/SegundoSemestre/MLOps/EDA_trabajofinal/mlruns/4', creation_time=1776041244663, experiment_id='4', last_update_time=1776041244663, lifecycle_stage='active', name='stroke-prediction2', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='file:///C:/Users/Lorena/Desktop/ULore/SegundoSemestre/MLOps/EDA_trabajofinal/mlruns/3', creation_time=1776037198951, experiment_id='3', last_update_time=1776037198951, lifecycle_stage='active', name='stroke-prediction1', tags={}, trace_location=None, workspace='default'>]

In [26]:
experiment_name = "stroke-prediction2"
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='file:///C:/Users/Lorena/Desktop/ULore/SegundoSemestre/MLOps/EDA_trabajofinal/mlruns/4', creation_time=1776041244663, experiment_id='4', last_update_time=1776041244663, lifecycle_stage='active', name='stroke-prediction2', tags={}, trace_location=None, workspace='default'>

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, mean_squared_error, mean_absolute_error, r2_score


with mlflow.start_run(run_name="3. RL oversampler"):
    max_iter=2000
    class_weight='balanced'
    model_LR = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=max_iter,class_weight=class_weight))
    ])
    #Tags
    mlflow.set_tag("problem_type", "binary classification")
    mlflow.set_tag("model_family", "LR")
    mlflow.set_tag("dataset", "healthcare-dataset-stroke-data")
    
    #Params
    mlflow.log_param("n_iter", max_iter)
    mlflow.log_param("class_weight", class_weight)

    
    #Train + predict
    #from imblearn.over_sampling import RandomOverSampler
    
    #ros = RandomOverSampler()
    #X_res, y_res = ros.fit_resample(X_train, y_train)
    
    model_LR.fit(X_train, y_train)
    #model_LR.fit(X_res, y_res)

    y_pred = model_LR.predict(X_test)
    y_proba = model_LR.predict_proba(X_test)[:, 1]
    report = classification_report(y_test, y_pred, output_dict=True)
    
    #metrics

    auc = roc_auc_score(y_test, y_proba)

    print(classification_report(y_test, y_pred))

    # 🔹 Log metrics
    mlflow.log_metric("roc_auc", auc)
    # 🔹 métricas globales
    mlflow.log_metric("accuracy", report["accuracy"])
    mlflow.log_metric("f1_macro", report["macro avg"]["f1-score"])
    mlflow.log_metric("f1_weighted", report["weighted avg"]["f1-score"])

    # 🔹 métricas clase positiva (stroke=1)
    mlflow.log_metric("precision_class_1", report["1"]["precision"])
    mlflow.log_metric("recall_class_1", report["1"]["recall"])
    mlflow.log_metric("f1_class_1", report["1"]["f1-score"])

    # 🔹 Log modelo
    mlflow.sklearn.log_model(model_LR, "model")

2026/04/12 19:49:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


              precision    recall  f1-score   support

           0       0.99      0.74      0.85       972
           1       0.14      0.80      0.24        50

    accuracy                           0.75      1022
   macro avg       0.56      0.77      0.54      1022
weighted avg       0.94      0.75      0.82      1022



2026/04/12 19:49:12 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\Lorena\Desktop\ULore\SegundoSemestre\MLOps\EDA_trabajofinal
2026/04/12 19:49:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/12 19:49:13 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\Lorena\Desktop\ULore\SegundoSemestre\MLOps\EDA_trabajofinal
2026/04/12 19:49:13 INFO mlflow.utils.environment: Detected uv project at c:\Users\Lorena\Desktop\ULore\SegundoSemestre\MLOps\EDA_trabajofinal. Attempting to export requirements via 'uv export'.
2026/04/12 19:49:13 INFO mlflow.utils.uv_utils: Exported

🏃 View run 3. RL oversampler at: http://127.0.0.1:5001/#/experiments/4/runs/e67ec8f0058d45f59cc0faec3db001f5
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/4


Probando con RandomForest

In [30]:
from sklearn.metrics import classification_report, roc_auc_score, mean_squared_error, mean_absolute_error, r2_score


with mlflow.start_run(run_name="5. RF 500"):
    n_estimators=500
    model_RF = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=n_estimators, random_state=42))
    ])
    #Tags
    mlflow.set_tag("problem_type", "binary classification")
    mlflow.set_tag("model_family", "RF")
    mlflow.set_tag("dataset", "healthcare-dataset-stroke-data")
    
    #Params
    mlflow.log_param("n_estimators", n_estimators)
    

    
    #Train + predict
    
    
    model_RF.fit(X_train, y_train)
    

    y_pred = model_RF.predict(X_test)
    y_proba = model_RF.predict_proba(X_test)[:, 1]
    report = classification_report(y_test, y_pred, output_dict=True)
    
    #metrics

    auc = roc_auc_score(y_test, y_proba)

    print(classification_report(y_test, y_pred))

    # 🔹 Log metrics
    mlflow.log_metric("roc_auc", auc)
    # 🔹 métricas globales
    mlflow.log_metric("accuracy", report["accuracy"])
    mlflow.log_metric("f1_macro", report["macro avg"]["f1-score"])
    mlflow.log_metric("f1_weighted", report["weighted avg"]["f1-score"])

    # 🔹 métricas clase positiva (stroke=1)
    mlflow.log_metric("precision_class_1", report["1"]["precision"])
    mlflow.log_metric("recall_class_1", report["1"]["recall"])
    mlflow.log_metric("f1_class_1", report["1"]["f1-score"])

    # 🔹 Log modelo
    mlflow.sklearn.log_model(model_RF, "model")

              precision    recall  f1-score   support

           0       0.95      1.00      0.97       972
           1       0.33      0.04      0.07        50

    accuracy                           0.95      1022
   macro avg       0.64      0.52      0.52      1022
weighted avg       0.92      0.95      0.93      1022



2026/04/12 19:53:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/12 19:53:02 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\Lorena\Desktop\ULore\SegundoSemestre\MLOps\EDA_trabajofinal
2026/04/12 19:53:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/12 19:53:03 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\Lorena\Desktop\ULore\SegundoSemestre\MLOps\EDA_trabajofinal
2026/04/12 19:53:03 INFO mlflow.utils.environment: Detected uv project at c:\Users\Lorena\Desktop\ULore\SegundoSemestre\MLOps\EDA_trabajofinal. 

🏃 View run 5. RF 500 at: http://127.0.0.1:5001/#/experiments/4/runs/dd99af686f5241f0b6e416c436693c66
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/4


En el contexto del dataset, es importante que el modelo tenga alta sensibilidad hacia los casos positivos. Debido al alto desbalance del dataset el accuracy no da información suficiente. Por lo que las métricas principales para seleccionar el mejor modelo son el Recall: De todos los pacientes que SÍ tienen stroke, ¿cuántos detecto?, Precisión: De los que predigo como stroke, ¿cuántos realmente lo son? y el F1-score: Balance entre precision y recall.

Recall clase 1:	detectar pacientes en riesgo
Precision clase 1: no generar falsas alarmas
F1 clase 1:	balance clínicamente razonable